# Encoding in Quantum Machine Learning
### Homework 2 — Angle Encoding and Amplitude Encoding with PennyLane

## Part 1: Generate the dataset

In [17]:
import numpy as np

X = np.array([
    [0.1, 0.5],
    [0.7, 0.2],
    [0.3, 0.8],
    [0.9, 0.4]
])

print(X)

[[0.1 0.5]
 [0.7 0.2]
 [0.3 0.8]
 [0.9 0.4]]


## Part 2: Implement Angle Encoding

In angle encoding, each feature value from the data point is used as the rotation angle of a quantum gate applied to one qubit. Here we use the RY gate, which rotates the qubit state around the Y axis of the Bloch sphere by the given angle.

According to the paper we read in class (I will refer to it as Zang et al), in angle encoding the input features are fed into the circuit as rotation gate angles (RX or RY) on their respective qubit, producing a product state across all qubits.

In [14]:
import pennylane as qml

#2 qubit simulator, one qubit per feature
dev = qml.device("default.qubit", wires=2)

@qml.qnode(dev)
def angle_encoding(x):
    #rotate qubit 0 by the first feature value
    qml.RY(x[0], wires=0)
    #rotate qubit 1 by the second feature value
    qml.RY(x[1], wires=1)
    return qml.state()

# test with the first data point
state = angle_encoding(X[0])
print("Quantum state for X[0] =", X[0])
print(state)

Quantum state for X[0] = [0.1 0.5]
[0.96770153+0.j 0.24709477+0.j 0.04842544+0.j 0.01236504+0.j]


## Part 3: Visualize the circuit

In [15]:
print(qml.draw(angle_encoding)(X[0]))

0: ──RY(0.10)─┤  State
1: ──RY(0.50)─┤  State


### Questions - Part 3

**How many qubits are used in this circuit?**

The circuit uses 2 qubits, one for each feature in the input vector. This is consistent with what Zang et al. (2025) describe: in angle encoding the number of qubits equals the number of features (n = N).

**What do the rotation angles represent in the encoding?**

Each rotation angle represents the value of one feature from the classical data point. The RY gate rotates the qubit state around the Y axis of the Bloch sphere by that angle, mapping the numerical value into a quantum superposition. A value close to 0 leaves the qubit near the |0> state, while a larger angle pushes it toward |1> or into a superposition between the two.

## Part 4: Experiment - Apply the circuit to all data points

In [16]:
#encoding circuit on every sample and print the resulting quantum state
print("Quantum states for each data point:")
for i, x in enumerate(X):
    state = angle_encoding(x)
    print(f"X[{i}] = {x}")
    print(f"state = {state}\n")

Quantum states for each data point:
X[0] = [0.1 0.5]
state = [0.96770153+0.j 0.24709477+0.j 0.04842544+0.j 0.01236504+0.j]

X[1] = [0.7 0.2]
state = [0.93467976+0.j 0.09378079+0.j 0.34118475+0.j 0.03423266+0.j]

X[2] = [0.3 0.8]
state = [0.91071847+0.j 0.38504559+0.j 0.13764163+0.j 0.05819395+0.j]

X[3] = [0.9 0.4]
state = [0.88249811+0.j 0.17889122+0.j 0.42629518+0.j 0.08641431+0.j]



### Analysis - Part 4

**Does the circuit change for each data point?**

The structure of the circuit stays the same for every sample: it always has two RY gates, one per qubit. What changes are the rotation angles, which take the values from the input data.

**How does the final quantum state change?**

Each data point produces a different quantum state because the rotation angles are different. The four amplitudes of the 2-qubit state shift with every sample, meaning each data point gets a unique encoding in the quantum state space. This is the point of angle encoding: the Bloch sphere position of each qubit changes with the input, giving the model distinguishable representations for different samples.

## Part 5: Reflection

**Why is encoding a fundamental step in Quantum Machine Learning?**

Quantum circuits can only operate on quantum states, they have no direct access to classical numbers. Encoding is the bridge that translates classical data into a quantum representation the circuit can actually process. Without this step, it is simply not possible to input classical data into a QML model. Its also important to remark that encoding part significantly affects the performance of a QML model, and a poor choice can limit accuracy regardless of how well the rest of the circuit is designed.

**What limitations does angle encoding have?**

- It scales linearly with the number of features: one qubit per feature. With 100 features you need 100 qubits, which is far beyond what current quantum hardware can support reliably.
- The rotation angles are bounded, so input data needs to be normalized carefully to avoid losing information in the mapping.
- It doesnt take advantage of the exponential capacity of the Hilbert space. Amplitude encoding can encode way more N features using less qubits
- Zang et al. also found that simple angle encoding tends to deliver average results across different datasets and model configurations, and is never the top performing encoding in their benchmarks.

**What would happen if the dataset had 100 features?**

With angle encoding, 100 features would require 100 qubits and 100 RY gates. Simulating this classically is extremely expensive because the quantum state would have 2^100 amplitudes. On real quantum hardware, this is also very hard to implement given that current NISQ devices have limited qubit counts and high error rates. This one of the main problemas

## Bonus: Amplitude Encoding

In amplitude encoding, the classical feature values are loaded directly as the amplitudes of the quantum state (given by the tensor product), rather than as rotation angles.

The key difference is efficiency. With n qubits, the quantum state has $2^n$ amplitudes, so you can encode $2^n$ features using only n qubits. This is a logarithmic relationship: for example, 8 features only need 3 qubits, and 1024 features would only need 10 qubits.

One important requirement: the input vector must be normalized (the sum of the squared values must equal 1) to form a valid quantum state.

### How is amplitude encoding different from angle encoding?

The main difference is where the data ends up in the quantum state:

- In angle encoding, each feature value becomes a rotation angle applied to one qubit via an RY gate. The data lives in the rotation parameters of individual gates. You need one qubit per feature, so the circuit grows linearly with the number of features.

- In amplitude encoding, the feature values are placed directly into the amplitudes of the quantum state. A state of n qubits has $2^n$ amplitudes, so you can encode up to $2^n$ features using only n qubits. This is the exponential advantage that quantum computing can offer for data representation.

However, amplitude encoding is not always better in practice. As shown in Zang et al. (2025), it tends to perform well on large datasets (like the Malware and MNIST datasets they tested), but underperforms on small datasets (like WDBC with only 569 samples). This is because preparing the exact amplitude state requires a deeper circuit with more controlled RY rotation gates, and with fewer training samples the model may not benefit enough to justify that extra complexity.